# Hyperparameter Optimization v2 — Corrected Methodology

**Why a v2**: `hyperparameter_tuning.ipynb` picked its winner by comparing
validation loss **across different `beta_max` values** and picked the lowest
beta in the grid (0.003) — but that comparison is invalid. `beta_max` directly
multiplies the KL term inside the very loss being compared, so a smaller beta
*mechanically* produces a smaller total loss almost regardless of model
quality. Fully training that "winner" made every real detection metric worse
(`cc1_test` F1 0.618 -> 0.574, `drift_cc2` F1 0.276 -> 0.244) — confirming the
comparison was measuring the wrong thing.

**Corrected methodology, in two separate phases that never compare loss
across different beta values:**

- **Phase A — pick beta by the collapse boundary, not by loss.** This is
  exactly the criterion `train_vae.ipynb` originally used ("largest beta_max
  that does not collapse the posterior = strongest valid regularization"),
  just re-run with a FINER grid than the original 4-point one — the original
  only tested {1.0, 0.1, 0.01, 0.001}; nothing between 0.01 and 0.1 was ever
  checked, and that is exactly where the true collapse boundary sits.
- **Phase B — tune latent_dim within that ONE fixed beta.** Comparing
  validation loss across different `latent_dim` values, at a FIXED beta, is a
  fair comparison (beta doesn't rescale the loss differently between these
  runs) — this is the same logic the original ablation used, just over a
  finer grid than {8, 16, 32}.
- **Phase C — full-train the winner, evaluate with the exact same protocol as
  `vae_eval.ipynb`, compare honestly against the deployed model.**

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle, os, time
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                              recall_score, f1_score, precision_recall_curve)

torch.manual_seed(42)
np.random.seed(42)

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')
OUT_DIR   = os.path.join(BASE, 'experiments')

DEVICE = torch.device('cpu')
ALL_SETS = ['cc1_test', 'drift_cc2']

HIDDEN1, HIDDEN2 = 64, 32
WARMUP_EPOCHS = 10
SCREEN_EPOCHS, SCREEN_PATIENCE = 40, 8
FULL_MAX_EPOCHS, FULL_PATIENCE = 300, 20
LR, BATCH_SIZE, CLIP = 1e-3, 512, 20.0
COLLAPSE_KL_THRESH = 0.05

BETA_SEARCH_LATENT_DIM = 16   # fixed reference, same convention as train_vae.ipynb
BETA_GRID_FINE = [0.1, 0.07, 0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]
LATENT_GRID = [8, 12, 16, 20, 24, 28, 32, 40, 48]

print(f'Phase A beta grid (fixed latent_dim={BETA_SEARCH_LATENT_DIM}): {BETA_GRID_FINE}')
print(f'Phase B latent_dim grid: {LATENT_GRID}')

Phase A beta grid (fixed latent_dim=16): [0.1, 0.07, 0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]
Phase B latent_dim grid: [8, 12, 16, 20, 24, 28, 32, 40, 48]


## Step 1 — Load data (same sanity guard as `train_vae.ipynb`)

In [2]:
raw = {name: np.load(os.path.join(DATA_DIR, f'X_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
labels = {name: np.load(os.path.join(DATA_DIR, f'y_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
data = {name: np.clip(X, -CLIP, CLIP).astype(np.float32) for name, X in raw.items()}
INPUT_DIM = data['cc1_train'].shape[1]

for name, X in raw.items():
    assert not np.isnan(X).any() and not np.isinf(X).any(), f'{name}: NaN/Inf found'
assert (labels['cc1_train'] == 0).all()

X_train_t = torch.from_numpy(data['cc1_train'])
X_val_t   = torch.from_numpy(data['cc1_val'])
print(f'INPUT_DIM={INPUT_DIM}  X_train={X_train_t.shape}  X_val={X_val_t.shape}')
print('Sanity checks passed.')

INPUT_DIM=26  X_train=torch.Size([154198, 26])  X_val=torch.Size([21573, 26])
Sanity checks passed.


## Step 2 — VAE class + training loop (identical to `train_vae.ipynb`)

In [3]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

def vae_loss(recon, x, mu, logvar, beta):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon_loss + beta * kl, recon_loss, kl

def train_vae(latent_dim, beta_max, max_epochs, patience, seed=42):
    torch.manual_seed(seed)
    model = VAE(INPUT_DIM, HIDDEN1, HIDDEN2, latent_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
    best_val, best_state, patience_ctr = float('inf'), None, 0
    final_kl, final_recon = None, None

    for epoch in range(max_epochs):
        beta = min(1.0, (epoch + 1) / WARMUP_EPOCHS) * beta_max
        model.train()
        for (xb,) in loader:
            opt.zero_grad()
            recon, mu, logvar = model(xb)
            loss, rloss, kl = vae_loss(recon, xb, mu, logvar, beta)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            recon, mu, logvar = model(X_val_t)
            vloss, vrecon, vkl = vae_loss(recon, X_val_t, mu, logvar, beta)
        final_kl, final_recon = vkl.item(), vrecon.item()
        if vloss.item() < best_val - 1e-6:
            best_val, best_state, patience_ctr = vloss.item(), {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, final_kl, final_recon, epoch + 1

print('Training function defined.')

Training function defined.


## Phase A — Find the true collapse boundary (fine beta grid, fixed latent_dim=16)

**Selection rule: largest beta that does NOT collapse the posterior**
(`final_kl >= 0.05`) — a qualitative check, never a loss comparison across
beta values. The original project only tested {1.0, 0.1, 0.01, 0.001} and
found 0.1 collapsed, 0.01 didn't — this fine grid fills in the gap between
them to find the actual boundary.

In [4]:
beta_search = {}
for beta_max in BETA_GRID_FINE:
    t0 = time.time()
    _, best_val, final_kl, final_recon, n_epochs = train_vae(BETA_SEARCH_LATENT_DIM, beta_max, SCREEN_EPOCHS, SCREEN_PATIENCE)
    elapsed = time.time() - t0
    collapsed = final_kl < COLLAPSE_KL_THRESH
    beta_search[beta_max] = {'best_val': best_val, 'final_kl': final_kl, 'final_recon': final_recon, 'collapsed': collapsed}
    print(f'  beta={beta_max:.3f}  val_loss={best_val:.4f}  final_kl={final_kl:.4f}  final_recon={final_recon:.4f}  '
          f'{"COLLAPSED" if collapsed else "ok"}  ({elapsed:.0f}s)')

non_collapsed = [b for b, r in beta_search.items() if not r['collapsed']]
assert non_collapsed, 'Every beta candidate collapsed.'
BEST_BETA = max(non_collapsed)
print(f'\nTrue collapse boundary found. Largest non-collapsing beta_max = {BEST_BETA}')
print(f'(Original coarse search picked 0.01 — this refined grid {"confirms the same value" if BEST_BETA == 0.01 else f"finds a stronger valid value: {BEST_BETA}"})')

  beta=0.100  val_loss=1.0779  final_kl=0.0179  final_recon=1.0773  COLLAPSED  (81s)
  beta=0.070  val_loss=0.8356  final_kl=4.1346  final_recon=0.5462  ok  (80s)
  beta=0.050  val_loss=0.7222  final_kl=5.9424  final_recon=0.4251  ok  (80s)
  beta=0.030  val_loss=0.5345  final_kl=10.5553  final_recon=0.2178  ok  (80s)
  beta=0.020  val_loss=0.4253  final_kl=13.7583  final_recon=0.1501  ok  (83s)
  beta=0.015  val_loss=0.3553  final_kl=15.8497  final_recon=0.1176  ok  (89s)
  beta=0.010  val_loss=0.2758  final_kl=19.1282  final_recon=0.0845  ok  (89s)
  beta=0.007  val_loss=0.2213  final_kl=22.7098  final_recon=0.0624  ok  (82s)
  beta=0.005  val_loss=0.1751  final_kl=25.4560  final_recon=0.0478  ok  (81s)
  beta=0.003  val_loss=0.1247  final_kl=30.5535  final_recon=0.0330  ok  (81s)
  beta=0.001  val_loss=0.0568  final_kl=40.8131  final_recon=0.0160  ok  (90s)

True collapse boundary found. Largest non-collapsing beta_max = 0.07
(Original coarse search picked 0.01 — this refined grid f

## Phase B — Latent-dimension ablation at the FIXED, correctly-chosen beta

This comparison IS valid — beta is now held constant, so validation loss
differences reflect only the latent_dim change, same logic the original
ablation used, just over a finer grid.

In [5]:
latent_results = {}
for latent_dim in LATENT_GRID:
    t0 = time.time()
    _, best_val, final_kl, final_recon, n_epochs = train_vae(latent_dim, BEST_BETA, SCREEN_EPOCHS, SCREEN_PATIENCE)
    elapsed = time.time() - t0
    collapsed = final_kl < COLLAPSE_KL_THRESH
    latent_results[latent_dim] = {'best_val': best_val, 'final_kl': final_kl, 'collapsed': collapsed}
    print(f'  latent_dim={latent_dim:3d}  val_loss={best_val:.4f}  final_kl={final_kl:.4f}  {"COLLAPSED" if collapsed else "ok"}  ({elapsed:.0f}s)')

valid_latents = {k: v for k, v in latent_results.items() if not v['collapsed']}
BEST_LATENT = min(valid_latents, key=lambda k: valid_latents[k]['best_val'])
print(f'\nBest latent_dim at beta={BEST_BETA}: {BEST_LATENT}  (val_loss={valid_latents[BEST_LATENT]["best_val"]:.4f})')
print(f'(Original ablation, at beta=0.01, picked latent_dim=32)')

  latent_dim=  8  val_loss=0.8478  final_kl=4.1968  ok  (89s)
  latent_dim= 12  val_loss=0.8156  final_kl=4.2012  ok  (86s)
  latent_dim= 16  val_loss=0.8356  final_kl=4.1346  ok  (80s)
  latent_dim= 20  val_loss=0.8261  final_kl=4.1983  ok  (83s)
  latent_dim= 24  val_loss=0.9430  final_kl=2.8962  ok  (87s)
  latent_dim= 28  val_loss=0.9525  final_kl=2.3872  ok  (93s)
  latent_dim= 32  val_loss=1.0525  final_kl=0.7602  ok  (93s)
  latent_dim= 40  val_loss=1.0616  final_kl=0.4737  ok  (89s)
  latent_dim= 48  val_loss=1.0790  final_kl=0.1000  ok  (33s)

Best latent_dim at beta=0.07: 12  (val_loss=0.8156)
(Original ablation, at beta=0.01, picked latent_dim=32)


## Phase C — Full training of the corrected winning configuration

In [6]:
print(f'Full training: latent_dim={BEST_LATENT}, beta_max={BEST_BETA} ...')
t0 = time.time()
tuned_model, tuned_val_loss, tuned_final_kl, tuned_final_recon, tuned_epochs = train_vae(
    BEST_LATENT, BEST_BETA, FULL_MAX_EPOCHS, FULL_PATIENCE)
elapsed = time.time() - t0
print(f'Done in {elapsed/60:.1f} min.  epochs={tuned_epochs}  val_loss={tuned_val_loss:.4f}  final_kl={tuned_final_kl:.4f}')
print(f'({"WARNING: looks collapsed" if tuned_final_kl < COLLAPSE_KL_THRESH else "non-trivial, ok"})')

Full training: latent_dim=12, beta_max=0.07 ...
Done in 7.5 min.  epochs=213  val_loss=0.7670  final_kl=4.4247
(non-trivial, ok)


## Step 3 — Full evaluation, identical protocol to `vae_eval.ipynb`

In [7]:
with torch.no_grad():
    mse_train = tuned_model.anomaly_score(X_train_t).numpy()
    mse_val = tuned_model.anomaly_score(X_val_t).numpy()
mu_train, sigma_train = float(mse_train.mean()), float(mse_train.std())
val_p99 = float(np.percentile(mse_val, 99))
print(f'Tuned model: mu_train={mu_train:.5f}  sigma_train={sigma_train:.5f}  val_p99 threshold={val_p99:.5f}')

def evaluate(scores, y_true, threshold):
    pred = (scores > threshold).astype(int)
    p, r, _ = precision_recall_curve(y_true, scores)
    f1s = 2 * p * r / (p + r + 1e-12)
    oracle_f1 = float(f1s[np.argmax(f1s)])
    return {
        'auc_roc': roc_auc_score(y_true, scores), 'auc_pr': average_precision_score(y_true, scores),
        'precision': precision_score(y_true, pred, zero_division=0), 'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0), 'oracle_f1': oracle_f1,
    }

tuned_results = {}
mse_by_set = {}
for name in ALL_SETS:
    X_t = torch.from_numpy(data[name])
    with torch.no_grad():
        mse = tuned_model.anomaly_score(X_t).numpy()
    mse_by_set[name] = mse
    tuned_results[name] = evaluate(mse, labels[name], val_p99)

deployed_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))

print(f'\n{"set":12s} {"model":10s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s} {"OracleF1":>9s}')
for name in ALL_SETS:
    t = tuned_results[name]
    d_auc = deployed_eval['auc'][name]
    d_pr = deployed_eval['precision_recall'][name]['val_p99']
    d_oracle = deployed_eval['oracle_ceiling'][name]['f1']
    print(f'{name:12s} {"tuned_v2":10s} {t["auc_pr"]:8.4f} {t["auc_roc"]:9.4f} {t["f1"]:7.3f} {t["precision"]:10.3f} {t["recall"]:8.3f} {t["oracle_f1"]:9.4f}')
    print(f'{name:12s} {"deployed":10s} {d_auc["auc_pr"]:8.4f} {d_auc["auc_roc"]:9.4f} {d_pr["f1"]:7.3f} {d_pr["precision"]:10.3f} {d_pr["recall"]:8.3f} {d_oracle:9.4f}')
    print(f'  -> PR-AUC change: {t["auc_pr"]-d_auc["auc_pr"]:+.4f}   F1 change: {t["f1"]-d_pr["f1"]:+.3f}   Oracle-F1 change: {t["oracle_f1"]-d_oracle:+.4f}\n')

Tuned model: mu_train=0.35896  sigma_train=0.33280  val_p99 threshold=1.34637

set          model        PR-AUC   ROC-AUC      F1  Precision   Recall  OracleF1
cc1_test     tuned_v2     0.5161    0.8976   0.503      0.514    0.492    0.6029
cc1_test     deployed     0.6014    0.8763   0.618      0.630    0.605    0.6857
  -> PR-AUC change: -0.0853   F1 change: -0.115   Oracle-F1 change: -0.0828

drift_cc2    tuned_v2     0.1123    0.8663   0.107      0.058    0.722    0.2371
drift_cc2    deployed     0.4089    0.8812   0.276      0.168    0.772    0.5147
  -> PR-AUC change: -0.2966   F1 change: -0.169   Oracle-F1 change: -0.2776



## Step 4 — Per-fault-type recall, tuned vs. deployed

In [8]:
ft = {name: np.load(os.path.join(DATA_DIR, f'ft_{name}.npy'), allow_pickle=True) for name in ALL_SETS}
deployed_fault = deployed_eval['per_fault_recall']

print(f'{"set":12s} {"fault_type":14s} {"n":>5s} {"deployed recall":>16s} {"tuned_v2 recall":>16s}')
for name in ALL_SETS:
    pred = (mse_by_set[name] > val_p99).astype(int)
    types_present = sorted({v for v in ft[name] if isinstance(v, str)})
    for ftype in types_present:
        mask = ft[name] == ftype
        n = int(mask.sum())
        rec_tuned = pred[mask].mean() if n > 0 else float('nan')
        rec_deployed = deployed_fault[name][ftype]['recall']
        print(f'{name:12s} {ftype:14s} {n:5d} {rec_deployed:16.3f} {rec_tuned:16.3f}')
    print()

set          fault_type         n  deployed recall  tuned_v2 recall
cc1_test     cpu               88            0.580            0.170
cc1_test     memory            76            0.763            0.750
cc1_test     pod-failure       92            0.500            0.587

drift_cc2    cpu              207            0.908            0.899
drift_cc2    memory           414            0.713            0.650
drift_cc2    pod-failure       99            0.737            0.657



## Step 5 — Save (does NOT overwrite the deployed model)

In [9]:
save_results = {
    'beta_search': beta_search,
    'best_beta': BEST_BETA,
    'latent_search': latent_results,
    'best_config': {'latent_dim': BEST_LATENT, 'beta_max': BEST_BETA},
    'tuned_model_meta': {
        'input_dim': INPUT_DIM, 'hidden1': HIDDEN1, 'hidden2': HIDDEN2, 'latent_dim': BEST_LATENT,
        'beta_max': BEST_BETA, 'clip': CLIP, 'mu_train': mu_train, 'sigma_train': sigma_train,
        'val_p99': val_p99, 'epochs_trained': tuned_epochs,
    },
    'tuned_results': tuned_results,
    'deployed_comparison': {
        name: {'auc': deployed_eval['auc'][name], 'precision_recall': deployed_eval['precision_recall'][name]['val_p99'],
               'oracle_f1': deployed_eval['oracle_ceiling'][name]['f1']}
        for name in ALL_SETS
    },
}
out_path = os.path.join(OUT_DIR, 'hyperparameter_tuning_v2_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

model_path = os.path.join(OUT_DIR, 'vae_cc1_tuned_v2.pt')
torch.save(tuned_model.state_dict(), model_path)
print(f'Tuned model weights saved -> {model_path}  (NOT deployed - models/vae_cc1.pt is unchanged)')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\hyperparameter_tuning_v2_results.pkl
Tuned model weights saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\vae_cc1_tuned_v2.pt  (NOT deployed - models/vae_cc1.pt is unchanged)


## How to read this

- **Phase A/B never compare loss across different beta values** — that was
  the exact flaw in `hyperparameter_tuning.ipynb` (v1). Beta is chosen by a
  qualitative collapse check; latent_dim is chosen by loss ONLY within that
  one fixed beta.
- **Step 3 is the honest verdict**: does this corrected search find a
  genuinely better configuration than the deployed model? If `BEST_BETA`
  comes back as 0.01 and `BEST_LATENT` as 32 (the original config), that is
  itself a strong, legitimate finding — it means the original coarse search
  already found the right answer, and this is now confirmed with a much
  finer grid rather than assumed. If a different configuration wins AND
  beats the deployed model's real detection metrics (not just its
  training loss), that is a genuine, reportable improvement.
- Nothing here overwrites `models/vae_cc1.pt` — promotion to "deployed" is a
  deliberate decision made only after Step 3's honest comparison, not an
  automatic step.